<a href="https://colab.research.google.com/github/python4recursion/bookcode/blob/main/CH07/7_huffman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CH07: Huffman Compression
import struct
class Bit:
    def __init__(self):
        self.bit = 0
        self.whichbit = 0 # which bit in the byte?
        self.byte = 0     # keep other bits in the byte
    def writeBit(self, fhd, b):
        self.bit = b
        if (self.whichbit == 0): # start a new byte
            self.byte = 0
        t = self.bit << (7 - self.whichbit)
        self.byte |= t
        if (self.whichbit == 7): # use up all bit
            fhd.write(struct.pack('B', self.byte))
        self.whichbit = (self.whichbit + 1) % 8
    def writeByte(self, onebyte, fhd):
        onebyte = ord(onebyte) # convert a letter to its ASCII value
        for i in range(0, 8):
            self.writeBit(fhd, int((0X80 & onebyte) != 0))
            onebyte <<= 1
    def readBit(self, fhd):
        if (self.whichbit == 0): # read a new byte
            self.byte = int.from_bytes(fhd.read(1), 'little')
        t = int(self.byte) >> (7 - self.whichbit)
        self.bit = int(t & 0X01)
        self.whichbit = (self.whichbit + 1) % 8
    def readByte(self, fhd): # call readBit 8 times
        onebyte = 0
        for i in range(0, 8):
            self.readBit(fhd)
            onebyte <<= 1
            onebyte |= self.bit
        return chr(onebyte)
    def padZero(self, fhd): # add zero bits to remaining bits
        while (self.whichbit != 0):
            self.writeBit(fhd, 0)
    def removeZero(self, fhd): # remove zero padding
        while (self.whichbit != 0):
            self.readBit(fhd)

In [2]:
class Tree:
    def __init__(self, ascii, occur):
        self.ascii = ascii
        self.occur = occur
        self.left = None
        self.right = None
    def merge(self, left, right):
        # make this the parent of two tree nodes
        self.left = left
        self.right = right
    def __str__(self): # used for printing
        return (self.ascii + ':' + str(self.occur))
    def print(self, level = 0, prefix = "Root:"):
        if self.right is not None:
            self.right.print(level + 1, "R--")
        print(" " * 6 * level + prefix, self.ascii, self.occur)
        if self.left is not None:
            self.left.print(level + 1, "L--")
    def postOrder(self, fhd, text = True, bitobj = None):
        if ((text == False) and (bitobj == None)):
            # this two conditions are not compatible
            return
        if (self == None):
            return
        if (self.left == None): # leaf node
            if (text == True):
              fhd.write('1')
              fhd.write(self.ascii)
            else:
              bitobj.writeBit(fhd, 1)
              bitobj.writeByte(self.ascii, fhd)
            return
        self.left.postOrder(fhd, text, bitobj)
        self.right.postOrder(fhd, text, bitobj)
        if (text == True):
          fhd.write('0')  # non-leaf node
        else:
          bitobj.writeBit(fhd, 0)
    def save(self, filename, text = True):
        if (text == True):
            bitobj = None
            fhd = open(filename, 'w')
            self.postOrder(fhd, text, bitobj)
            fhd.write('0') # end the tree
            fhd.write('\n')
        else:
            bitobj = Bit()
            fhd = open(filename, 'wb')
            self.postOrder(fhd, text, bitobj)
            bitobj.writeBit(fhd, 0)
            bitobj.padZero(fhd)
            bitobj.writeByte('\n', fhd)
        fhd.close()
    def getHeight(self):
        if (self == None):
            return 0
        if (self.left == None):
            leftheight = 0
        else:
            leftheight = self.left.getHeight()
        if (self.right == None):
            rightheight = 0
        else:
            rightheight = self.right.getHeight()
        if (leftheight > rightheight):
            return (leftheight + 1) # add one to include the root
        return (rightheight + 1)

In [3]:
class List:
    def __init__(self):
        self.tree = None
        self.next = None
    def addASCIIoccur(self, ascii, occur):
        t = Tree(ascii, occur)
        self.addTree(t, True)
    def addTree(self, tree, tosort):
        ln = List()
        ln.tree = tree
        if (self.next == None): # first node
            self.next = ln
            return
        if (tosort == False): # add at the beginning
            ln.next = self.next
            self.next = ln
            return
        occur = tree.occur
        if (occur < ((self.next).tree).occur):
            ln.next = self.next # ln becomes the first
            self.next = ln
            return
        # find where to insert ln
        p = self.next
        q = p.next
        while ((q != None) and ((q.tree.occur) < occur)):
            p = p.next
            q = q.next
        # It is possible that q is None, but that does not matter
        p.next = ln
        ln.next = q
    def delete(self): # delete the first node
        self.next = (self.next).next
    def print(self):
        p = self.next
        while (p != None):
            p.tree.print()
            p = p.next
    def __str__(self):
        p = self.next
        while (p != None):
            print(p.tree)
            p = p.next
        return ''

In [4]:
class Compressor:
    def __init__(self, infile, text_outfile, binary_outfile):
        self.infile = infile
        self.text_outfile = text_outfile
        self.binary_outfile = binary_outfile
        self.count = dict()
        self.length = 0
        self.tree = None
        self.code = None
        self.book = None
    def countOccur(self):
        fhd = open(self.infile, 'r')
        # key: letter; value: count
        self.content = fhd.read()  # read the entire file
        for i in range(0, len(self.content)):
            ascii = self.content[i]  # get the ASCII value
            if (ascii in self.count):  # already seen
                self.count[ascii] = self.count[ascii] + 1
            else:
                self.count[ascii] = 1  # first time
        fhd.close()
        self.length = len(self.content)
        print(f'length = {self.length}')
    def buildTree(self):
        ln = List()
        for ascii in self.count:
            ln.addASCIIoccur(ascii, self.count[ascii])
        while ((ln.next).next != None):
            # get the first two Tree nodes
            t1 = ln.next.tree
            t2 = ln.next.next.tree
            tp = Tree('-', t1.occur + t2.occur) # parent
            # make them siblings
            tp.merge(t1, t2)
            # remove the first two from the list
            ln.delete()
            ln.delete()
            # insert back in the ascending order (True)
            ln.addTree(tp, True)
        self.tree = ln.next.tree
    def codeHelper(self, tree, ind):
      if (tree.left == None): # leaf node
        self.book[tree.ascii] = self.code[0:ind]
        return
      # left subtree
      self.code[ind] = '0'
      self.codeHelper(tree.left, ind + 1)
      # right subtree
      self.code[ind] = '1'
      self.codeHelper(tree.right, ind + 1)
    def buildCodeBook(self):
      h = self.tree.getHeight()
      self.code = [' ' for i in range(0, h)]
      self.book = dict()
      self.codeHelper(self.tree, 0)
      print(f'codebook = {self.book}')
    def compress(self):
      self.countOccur()
      self.buildTree()
      self.tree.print()
      self.tree.save(self.text_outfile, True)
      self.tree.save(self.binary_outfile, False)
      self.buildCodeBook()
      # write the original file's length
      # text output
      text_fhd = open(self.text_outfile, 'a') # 'a': append
      text_fhd.write(str(len(self.content)))
      text_fhd.write('\n')
      # binary output
      binary_fhd = open(self.binary_outfile, 'ab') # 'b': binary
      bitobj = Bit()
      binary_fhd.write(struct.pack('i', len(self.content))) # 4-byte int
      bitobj.writeByte('\n', binary_fhd)
      # write the content
      for i in range(0, len(self.content)):
          code = self.book[self.content[i]]
          for j in range(0, len(code)):
              text_fhd.write(code[j])
              bitobj.writeBit(binary_fhd, int(code[j]))
      bitobj.padZero(binary_fhd)
      text_fhd.close()
      binary_fhd.close()

In [5]:
class Decompressor:
    def __init__(self, text_infile, binary_infile, outfile):
        self.text_infile = text_infile
        self.binary_infile = binary_infile
        self.outfile = outfile
        self.text_tree = None
        self.binary_tree = None
    def buildTextTree(self, text_fhd):
        ln = List()
        finishedtree = False
        numOne  = 0
        numZero = 0
        while (finishedtree == False):
            ctrl = text_fhd.read(1)
            if (ctrl == '1'):
                numOne = numOne + 1
                ascii = text_fhd.read(1) # read the letter
                tn = Tree(ascii, -1) # do not care about occur, use -1
                ln.addTree(tn, False)
            if (ctrl == '0'):
                numZero += 1
                # get the latest two trees
                tr1 = (ln.next).tree
                if (ln.next.next != None):
                    tr2 = ((ln.next).next).tree
                    tp = Tree('-', -1)
                    # do not care about the letter or occur of a non-leaf node
                    tp.merge(tr2, tr1) # careful about the order
                    ln.delete()
                    ln.delete()
                    ln.addTree(tp, False)
            if (numZero == numOne):
                finishedtree = True
        text_fhd.read(1) # remove '\n'
        self.text_tree = ln.next.tree
    def buildBinaryTree(self, bitobj, binary_fhd):
        ln = List()
        finishedtree = False
        numOne  = 0
        numZero = 0
        while (finishedtree == False):
            bitobj.readBit(binary_fhd)
            ctrl = bitobj.bit
            if (ctrl == 1):
                numOne = numOne + 1
                ascii = bitobj.readByte(binary_fhd)
                tn = Tree(ascii, -1)
                ln.addTree(tn, False)
            if (ctrl == 0):
                numZero += 1
                # get the latest two trees
                tr1 = (ln.next).tree
                if (ln.next.next != None):
                    tr2 = ((ln.next).next).tree
                    tp = Tree('-', -1)
                    tp.merge(tr2, tr1) # careful about the order
                    ln.delete()
                    ln.delete()
                    ln.addTree(tp, False)
            if (numZero == numOne):
                finishedtree = True
        bitobj.removeZero(binary_fhd)
        binary_fhd.read(1) # remove '\n'
        self.binary_tree = ln.next.tree
    def decompress(self):
        # decompress the text format
        text_fhd = open(self.text_infile, 'r')
        self.buildTextTree(text_fhd)
        text_output=[]
        text_length = int(text_fhd.readline())
        print(f'text_length = {text_length}')
        tn = self.text_tree
        while (text_length > 0):
          code = text_fhd.read(1) # read one byte
          if (code == '0'):
              tn = tn.left
          if (code == '1'):
              tn = tn.right
          if (tn.left == None): # left node
              text_output.append(tn.ascii)
              tn = self.text_tree
              text_length -= 1
        text_fhd.close()
        # decompress the binary format
        binary_fhd = open(self.binary_infile, 'rb')
        bitobj = Bit()
        self.buildBinaryTree(bitobj, binary_fhd)
        binary_output = []
        binary_length = int.from_bytes(binary_fhd.read(4), 'little')
        print(f'binary_length = {binary_length}')
        binary_fhd.read(1) # read one byte, remove '\n'
        tn = self.binary_tree
        while (binary_length > 0):
            if (self.binary_tree.left == None):
              binary_output.append(tn.ascii)
              binary_length -= 1
            else:
              bitobj.readBit(binary_fhd)
              code = bitobj.bit
              if (code == 0):
                  tn = tn.left
              if (code == 1):
                  tn = tn.right
              if (tn.left == None): # leaf node
                  binary_output.append(tn.ascii)
                  tn = self.binary_tree
                  binary_length -= 1
        binary_fhd.close()
        # compare the output from text and binary
        if (text_output != binary_output):
            print('text and binary output are different')
            print(f'text   length = {len(text_output)}, {text_output}')
            print(f'binary length = {len(binary_output)}, {binary_output}')
        outfhd = open(self.outfile, 'w')
        outfhd.write(''.join(binary_output))
        outfhd.close()

In [6]:
import os
!apt-get install xxd
def test_compress(base_path):
  input_path = base_path + 'inputs/'
  expected_text_path = base_path + 'expected_text/'
  expected_binary_path = base_path + 'expected_binary/'
  if os.path.isdir(input_path):
    for filename in sorted(os.listdir(input_path)):
      if filename.startswith('input'):
        filepath = os.path.join(input_path, filename)
        if os.path.isfile(filepath):
          print(f"input      file: {filepath}")
          output_name = program_path + filename.replace('input', 'output')
          expected_name = filename.replace('input', 'expected')
          text_output = output_name + '.txt'
          binary_output = output_name + '.bin'
          text_expected = program_path + 'text_expected/' + expected_name + '.txt'
          binary_expected = program_path + 'binary_expected/' + expected_name + '.bin'
          print(f'text     output: {text_output}')
          print(f'binary   output: {binary_output}')
          print(f'text   expected: {text_expected}')
          print(f'binary expected: {binary_expected}')
          comp = Compressor(filepath, text_output, binary_output)
          comp.compress()
          print('compare text output')
          !diff -y -q "$text_output" "$text_expected"
          print('compare binary output')
          !diff -y -q "$binary_output" "$binary_expected"
          # !xxd -b "$binary_output"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xxd is already the newest version (2:8.2.3995-1ubuntu2.30).
xxd set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [7]:
def test_decompress(base_path):
  if os.path.isdir(base_path):
    for filename in sorted(os.listdir(base_path)):
      if (filename.startswith('output') and filename.endswith('.bin')):
        binary_compressed = os.path.join(base_path, filename)
        text_compressed = binary_compressed.replace('.bin', '.txt')
        output_name = binary_compressed.replace('.bin', '')
        output_name = output_name.replace('output', 'decompressed')
        print('\n-------')
        print(f"binary file: {binary_compressed}")
        print(f"text   file: {text_compressed}")
        print(f"output file: {output_name}")
        original_name = output_name.replace('decompressed', 'input')
        # original_name = original_name.replace('CH07 Huffman', 'CH07 Huffman/inputs')
        original_name = original_name.replace('bookcode/7', 'bookcode/7/inputs')
        print(f'original file: {original_name}')
        dec = Decompressor(text_compressed, binary_compressed, output_name)
        dec.decompress()
        print('compare decompressed with original file')
        print(f'diff output = {output_name}, original = {original_name}')
        !diff -y -q "$output_name" "$original_name"


In [8]:
!rm -f -r bookcode/
!git clone https://github.com/python4recursion/bookcode.git
!ls -F bookcode/CH07/
program_path = 'bookcode/CH07/'
test_compress(program_path)

Cloning into 'bookcode'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 259 (delta 69), reused 42 (delta 42), pack-reused 147 (from 2)
Receiving objects: 100% (259/259), 1.09 MiB | 7.55 MiB/s, done.
Resolving deltas: 100% (92/92), done.
7_huffman.ipynb  binary_expected/  inputs/  text_expected/
input      file: bookcode/CH07/inputs/input1
text     output: bookcode/CH07/output1.txt
binary   output: bookcode/CH07/output1.bin
text   expected: bookcode/CH07/text_expected/expected1.txt
binary expected: bookcode/CH07/binary_expected/expected1.bin
length = 240
                        R-- b 21
                  R-- - 42
                        L-- h 21
            R-- - 72
                        R-- c 18
                  L-- - 30
                        L-- f 12
      R-- - 135
            L-- e 63
Root: - 240
            R-- a 54
      L-- - 105
                  R-- g 27
            L-

In [9]:
test_decompress(program_path)


-------
binary file: bookcode/CH07/output1.bin
text   file: bookcode/CH07/output1.txt
output file: bookcode/CH07/decompressed1
original file: bookcode/CH07/input1
text_length = 240
binary_length = 240
compare decompressed with original file
diff output = bookcode/CH07/decompressed1, original = bookcode/CH07/input1
diff: bookcode/CH07/input1: No such file or directory

-------
binary file: bookcode/CH07/output10.bin
text   file: bookcode/CH07/output10.txt
output file: bookcode/CH07/decompressed10
original file: bookcode/CH07/input10
text_length = 26476
binary_length = 26476
compare decompressed with original file
diff output = bookcode/CH07/decompressed10, original = bookcode/CH07/input10
diff: bookcode/CH07/input10: No such file or directory

-------
binary file: bookcode/CH07/output11.bin
text   file: bookcode/CH07/output11.txt
output file: bookcode/CH07/decompressed11
original file: bookcode/CH07/input11
text_length = 1854
binary_length = 1854
compare decompressed with original file

In [10]:
'''
from google.colab import drive
#drive.mount('/content/drive/Shared with me', force_remount = True)
drive.mount('/content/drive/')
program_path = '/content/drive/My Drive/Colab Notebooks/Recursion Book/CH07 Huffman/'
'''

"\nfrom google.colab import drive\n#drive.mount('/content/drive/Shared with me', force_remount = True)\ndrive.mount('/content/drive/')\nprogram_path = '/content/drive/My Drive/Colab Notebooks/Recursion Book/CH07 Huffman/'\n"